## Model Training and Comparison

> **Training and comparative evaluation of Random Forest, XGBoost, and Gradient Boosting for NBA win prediction.**  
> Classification predicts `home_win`; regression predicts `point_differential` and can be converted back into a win signal.  
> **Input:** `data/3_features/features_nba_data_2000-01_2025-26.csv`  
> **Output:** trained model and preprocessor artifacts in `models/`.

---

**Pipeline**

1. Setup — imports, project modules, configuration, logging, and visual defaults
2. Dataset Overview — split shapes, target distribution, feature list, baselines, and walk-forward CV
3. Training — classification and regression model fitting
4. Classification Evaluation — metrics, ROC curves, and confusion-matrix diagnostics
5. Regression Evaluation — spread metrics, scatter plots, residuals, and derived win prediction
6. Feature Importance — top predictors and cross-model consistency
7. Error Analysis — misclassified games, large spread errors, and contextual segmentation
8. Conclusion — dynamic synthesis of metrics, feature signals, and model limitations

### 1. Setup & import from `src/`

* **What:** Imports required libraries, adds the project root to the system path, and sets up global configurations (logging, pandas display options, and Seaborn visual themes).
* **Why:** To establish a reproducible, clean, and properly configured environment before performing any data operations.
* **Reasoning/Choices:** Suppressing warnings ensures clean output for presentation. Setting global visual themes guarantees consistency across all plots. Centralizing colors and labels in constants makes future stylistic updates effortless.

In [ ]:
import logging
import shutil
import sys
import warnings
from pathlib import Path

# Set the project root directory and add it to the system path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))
# Suppress warning messages for cleaner output
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats
# Import metrics for evaluating model performance
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
import xgboost as xgb

# Import custom project modules for loading, preprocessing, training, and evaluation
from src.ml.data_loader import load_splits, FEATURES_PATH
from src.ml.preprocessing import (
    FEATURE_COLS,
    HOME_FEATURES,
    AWAY_FEATURES,
    DIFF_FEATURES,
    CONTEXTUAL_FEATURES,
    transform,
)
from src.ml.trainer import train_all, MODELS_DIR
from src.ml.evaluator import plot_roc_curves, predict_proba
from src.loader import load_config

# Load config
config = load_config()

# Configure the logging format to track notebook execution progress
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

# Set global pandas display options for better readability
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
# Apply visual theme to seaborn plots
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

# Notebook configuration constants
NOTEBOOK_VERSION = "1.0.0"
MODEL_NAMES      = ["random_forest", "xgboost", "gradient_boosting"]
MODEL_LABELS     = ["Random Forest", "XGBoost", "Gradient Boosting"]
MODEL_COLORS     = ["steelblue", "darkorange", "forestgreen"]

# Log initial notebook configuration info
log.info("Notebook v%s — initialized", NOTEBOOK_VERSION)
log.info("Project root : %s", ROOT)
log.info("Models dir   : %s", MODELS_DIR)
log.info("Features path: %s", FEATURES_PATH)

### 2. Dataset Overview

* **What:** Reviews model inputs, temporal splits, target distributions, baselines, and walk-forward validation setup.
* **Why:** To confirm that the training data is correctly structured before fitting any estimator.
* **Reasoning/Choices:** Dataset checks are separated from training so split leakage, target imbalance, or unexpected feature definitions can be identified early.

#### 2.1 Split Shapes, Target Distribution, Feature List

* **What:** Loads the training, validation, and testing datasets for both classification and regression tasks, then prints summary statistics (row counts, feature dimensions, and target variable distributions).
* **Why:** To verify data integrity and ensure splits were created correctly before initiating the training process.
* **Reasoning/Choices:** Explicitly printing class balances (home win rates) and target ranges (point differentials) acts as a sanity check to identify potential class imbalances or unexpected data scales early on.


In [ ]:
# --- Classification data splits ---
# Load training, validation, and testing sets for the classification task (predicting 'home_win')
(X_train_clf, y_train_clf), (X_val_clf, y_val_clf), (X_test_clf, y_test_clf) = load_splits("classification")

print("=" * 60)
print("Classification  (target: home_win)")
print("=" * 60)
# Output summary statistics for each classification split
for name, X, y in [("train", X_train_clf, y_train_clf),
                    ("val",   X_val_clf,   y_val_clf),
                    ("test",  X_test_clf,  y_test_clf)]:
    hw = y.mean()
    print(f"  {name:5s}  rows={len(y):>6d}  features={X.shape[1]:>3d}  "
          f"home_win_rate={hw:.3f}  away_win_rate={1-hw:.3f}")

# --- Regression data splits ---
# Load training, validation, and testing sets for the regression task (predicting 'point_differential')
(X_train_reg, y_train_reg), (X_val_reg, y_val_reg), (X_test_reg, y_test_reg) = load_splits("regression")

print()
print("=" * 60)
print("Regression  (target: point_differential)")
print("=" * 60)
# Output summary statistics for each regression split
for name, X, y in [("train", X_train_reg, y_train_reg),
                    ("val",   X_val_reg,   y_val_reg),
                    ("test",  X_test_reg,  y_test_reg)]:
    print(f"  {name:5s}  rows={len(y):>6d}  "
          f"pt_diff  mean={y.mean():+.2f}  std={y.std():.2f}  "
          f"[{y.min():.0f}, {y.max():.0f}]")

#### 2.2 Canonical Features List

* **What:** Iterates through and prints the canonical list of predictive features grouped by their logical categories (Home, Away, Differential, Contextual).
* **Why:** To provide transparent documentation of the exact input space the models will utilize.
* **Reasoning/Choices:** Explicitly highlighting the `is_covid_bubble` feature preemptively addresses domain-specific concerns regarding how data from the unique 2019-20 Orlando Bubble is handled by the pipeline.

In [ ]:
# --- Canonical feature list ---
# Group features for systematic display and verification
feature_groups = {
    f"Home rolling/seasonal  ({len(HOME_FEATURES)})": HOME_FEATURES,
    f"Away rolling/seasonal  ({len(AWAY_FEATURES)})": AWAY_FEATURES,
    f"Differential (home−away)  ({len(DIFF_FEATURES)})": DIFF_FEATURES,
    f"Contextual  ({len(CONTEXTUAL_FEATURES)})": CONTEXTUAL_FEATURES,
}

# Iterate through groups and list all feature columns
for group_name, cols in feature_groups.items():
    print(f"\n{'─'*55}")
    print(f"  {group_name}")
    print(f"{'─'*55}")
    for col in cols:
        # Highlight the special status of the COVID bubble feature
        marker = "  <- COVID context" if col == "is_covid_bubble" else ""
        print(f"  {col}{marker}")

# Print final count of all features used in the model
print(f"\n{'═'*55}")
print(f"  Total features: {len(FEATURE_COLS)}")
print(f"{'═'*55}")
print()
# Note regarding the inclusion of the Orlando Bubble data
print("NOTE: 'is_covid_bubble' is included as a feature—")
print("      the model will learn its relevance automatically")
print("      without excluding the games from the Orlando Bubble.")

### 2.3 Baseline calculation

* **What:** Computes the naive baseline accuracy by predicting the majority class (usually the Home Team) for all instances across all splits.
* **Why:** To establish a minimum performance threshold for the classification models.
* **Reasoning/Choices:** A machine learning model is only valuable if it outperforms simple heuristics. The majority-class rule is the standard benchmark for binary classification tasks.

In [ ]:
# --- 1. Classification Baseline ---
# Determine the majority class from the training set to establish a baseline for classification
if hasattr(y_train_clf, "value_counts"):
    majority_class = y_train_clf.value_counts().idxmax()
else:
    majority_class = 1 if np.mean(y_train_clf) >= 0.5 else 0

# Calculate accuracy scores for the training, validation, and test sets
BASELINE_ACC      = np.mean(y_train_clf == majority_class)
baseline_val_acc  = np.mean(y_val_clf == majority_class)
baseline_test_acc = np.mean(y_test_clf == majority_class)

print("Classification Baseline (Majority Class Prediction)")
print("-" * 55)
print(f"Train Baseline Accuracy (Majority Class '{majority_class}'): {BASELINE_ACC:.4f}")
print(f"Validation Baseline Accuracy:                      {baseline_val_acc:.4f}")
print(f"Test Baseline Accuracy:                          {baseline_test_acc:.4f}")

In [ ]:
# --- 2. Regression Baseline ---
# Use the training target mean as a naive prediction for regression
train_mean = y_train_reg.mean()

# Create constant prediction arrays based on the training mean
y_pred_val_base  = np.full(len(y_val_reg), train_mean)
y_pred_test_base = np.full(len(y_test_reg), train_mean)

print("\nRegression Baseline (Train Mean Prediction)")
print("-" * 55)
print(f"Train Target Mean used for predictions:        {train_mean:+.2f}")
print(f"Validation Baseline MAE:                       {mean_absolute_error(y_val_reg, y_pred_val_base):.4f}")
print(f"Test Baseline MAE:                             {mean_absolute_error(y_test_reg, y_pred_test_base):.4f}")

#### 2.4 Walk-Forward Cross-Validation

* **What:** Executes a time-series aware Walk-Forward Cross-Validation using explicit fold definitions and a localized data loading/preprocessing loop.
* **Why:** To rigorously assess model stability and generalization across different historical seasons without introducing data leakage.
* **Reasoning/Choices:** Walk-Forward validation is strictly required over standard K-Fold CV; randomly shuffling historical NBA data would allow "future" games to predict "past" games. The `SimpleImputer` is explicitly fitted *inside* the loop to simulate a real-world scenario with completely unseen validation data.

Because NBA data has a temporal structure, random shuffling would introduce data leakage. This section therefore uses **walk-forward cross-validation** by season: the training window grows progressively, and the validation season moves forward one season at a time.

**Folds (5 folds, using only the original 2000-01 to 2018-19 train block):**

| Fold | Train | Val |
|------|-------|-----|
| 1 | 2000-01 -> 2013-14 (14 seasons) | 2014-15 |
| 2 | 2000-01 -> 2014-15 (15 seasons) | 2015-16 |
| 3 | 2000-01 -> 2015-16 (16 seasons) | 2016-17 |
| 4 | 2000-01 -> 2016-17 (17 seasons) | 2017-18 |
| 5 | 2000-01 -> 2017-18 (18 seasons) | 2018-19 |

The 2018-19 season (fold 5 validation) is the final season in the original train block. The official validation and test seasons, 2019-20 to 2025-26, are never used during CV.

The CV models match the main training models and use the same default hyperparameters to keep the evaluation consistent.

In [ ]:
# --- Section 2.3: Walk-Forward Cross-Validation -----------------------------
# Reload the dataset explicitly so this section is self-contained.
df_cv = pd.read_csv(FEATURES_PATH)

# Local 42-feature list used for this CV experiment.
CV_FEATURE_COLS = [
    "home_net_rating_rolling_10",
    "home_avg_pts_last_10",
    "home_avg_ts_pct_last_10",
    "home_avg_pace_last_10",
    "home_winrate_last_10",
    "home_streak",
    "home_rest_days",
    "home_is_back_to_back",
    "home_fg_pct",
    "home_fg3_pct",
    "home_ft_pct",
    "home_ts_pct_zscore",
    "home_pace_zscore",
    "home_win_rate_prev_season",
    "away_net_rating_rolling_10",
    "away_avg_pts_last_10",
    "away_avg_ts_pct_last_10",
    "away_avg_pace_last_10",
    "away_winrate_last_10",
    "away_streak",
    "away_rest_days",
    "away_is_back_to_back",
    "away_fg_pct",
    "away_fg3_pct",
    "away_ft_pct",
    "away_ts_pct_zscore",
    "away_pace_zscore",
    "away_win_rate_prev_season",
    "net_rating_rolling_10_diff",
    "avg_pts_last_10_diff",
    "avg_ts_pct_last_10_diff",
    "avg_pace_last_10_diff",
    "winrate_last_10_diff",
    "streak_diff",
    "rest_days_diff",
    "fg_pct_diff",
    "fg3_pct_diff",
    "ft_pct_diff",
    "ts_pct_zscore_diff",
    "pace_zscore_diff",
    "win_rate_prev_season_diff",
    "is_back_to_back_diff",
]

# Convert boolean columns to integers before fitting scikit-learn and XGBoost models.
bool_cols = [col for col in ["home_is_back_to_back", "away_is_back_to_back"] if col in df_cv.columns]
df_cv[bool_cols] = df_cv[bool_cols].astype(int)

train_seasons = sorted(df_cv.loc[df_cv["split"] == "train", "season"].unique())

# Define walk-forward folds inside the original train block only.
FOLDS = [
    {
        "train_seasons": [season for season in train_seasons if season <= "2013-14"],
        "val_season": "2014-15",
    },
    {
        "train_seasons": [season for season in train_seasons if season <= "2014-15"],
        "val_season": "2015-16",
    },
    {
        "train_seasons": [season for season in train_seasons if season <= "2015-16"],
        "val_season": "2016-17",
    },
    {
        "train_seasons": [season for season in train_seasons if season <= "2016-17"],
        "val_season": "2017-18",
    },
    {
        "train_seasons": [season for season in train_seasons if season <= "2017-18"],
        "val_season": "2018-19",
    },
]

# Define the models compared in CV.
CV_MODELS = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    ),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv_results = {
    model_name: {"accuracy": [], "f1": [], "auc_roc": []}
    for model_name in CV_MODELS
}

print(f"{'Fold':<6} {'Model':<22} {'Accuracy':>10} {'F1':>10} {'AUC-ROC':>10}")
print("-" * 62)

for fold_idx, fold in enumerate(FOLDS, 1):
    train_mask = df_cv["season"].isin(fold["train_seasons"])
    val_mask = df_cv["season"] == fold["val_season"]

    X_train_fold = df_cv.loc[train_mask, CV_FEATURE_COLS]
    y_train_fold = df_cv.loc[train_mask, "home_win"]
    X_val_fold = df_cv.loc[val_mask, CV_FEATURE_COLS]
    y_val_fold = df_cv.loc[val_mask, "home_win"]

    # Fit the imputer only on the current fold's training data to avoid leakage.
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_fold)
    X_val_imp = imputer.transform(X_val_fold)

    for model_name, model in CV_MODELS.items():
        clf = model.__class__(**model.get_params())
        clf.fit(X_train_imp, y_train_fold)

        y_pred = clf.predict(X_val_imp)
        y_prob = clf.predict_proba(X_val_imp)[:, 1]

        acc = accuracy_score(y_val_fold, y_pred)
        f1 = f1_score(y_val_fold, y_pred, zero_division=0)
        auc = roc_auc_score(y_val_fold, y_prob)

        cv_results[model_name]["accuracy"].append(acc)
        cv_results[model_name]["f1"].append(f1)
        cv_results[model_name]["auc_roc"].append(auc)

        print(f"  {fold_idx}    {model_name:<22} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")

    print()

print("Walk-forward CV completed.")

#### 2.5 Walk Forward CV Summary

* **What:** Aggregates the cross-validation metrics into a summary DataFrame, reporting the mean and standard deviation ($\mu \pm \sigma$) for accuracy, F1, and AUC-ROC.
* **Why:** To summarize model stability in a concise, statistically sound format.
* **Reasoning/Choices:** Reporting $\mu \pm \sigma$ complies with academic and examination standards, proving the model's performance isn't heavily dependent on a single lucky data split.

In [ ]:
# --- Summary table: μ ± σ by model and metric -------------------------------

summary_rows = []
for model_name in CV_MODELS:
    row = {"Model": model_name}
    for metric in ["accuracy", "f1", "auc_roc"]:
        vals = np.array(cv_results[model_name][metric])
        row[f"{metric}_mean"] = vals.mean()
        row[f"{metric}_std"] = vals.std(ddof=1)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Model")

# Format combined columns as mean ± standard deviation.
display_df = pd.DataFrame(index=summary_df.index)
display_df["Accuracy (μ ± σ)"] = summary_df.apply(
    lambda row: f"{row['accuracy_mean']:.4f} ± {row['accuracy_std']:.4f}",
    axis=1,
)
display_df["F1 (μ ± σ)"] = summary_df.apply(
    lambda row: f"{row['f1_mean']:.4f} ± {row['f1_std']:.4f}",
    axis=1,
)
display_df["AUC-ROC (μ ± σ)"] = summary_df.apply(
    lambda row: f"{row['auc_roc_mean']:.4f} ± {row['auc_roc_std']:.4f}",
    axis=1,
)

print("Walk-Forward Cross-Validation Results (5 folds, seasons 2014-15 to 2018-19)\n")
display(
    display_df.style.set_caption(
        "μ = mean across the 5 folds, σ = sample standard deviation (ddof=1)"
    )
)

best_model = summary_df["accuracy_mean"].idxmax()
print(f"\nModel with the highest mean CV accuracy: {best_model}")
print("Note: a low σ indicates that the estimate is stable across seasons.")

#### 2.6 Walk-Forward Interpretation

* **What:** Interprets cross-validation means and standard deviations as evidence of model stability across unseen seasons.
* **Why:** To distinguish consistently generalizable performance from results driven by a favorable historical period.
* **Reasoning/Choices:** High variance across folds suggests sensitivity to NBA style changes, lockout seasons, or other season-level shifts. The 2019-20 COVID bubble remains excluded from these folds because it belongs to the official validation block.

### 3. Training

* **What:** Trains Random Forest, XGBoost, and Gradient Boosting models for both classification and regression tasks.
* **Why:** To compare whether direct win prediction or point-differential prediction better captures NBA game outcomes.
* **Reasoning/Choices:** The shared `train_all` pipeline saves each model and preprocessor consistently, reducing notebook boilerplate and preventing differences in preprocessing between algorithms.

#### 3.1 Resetting Models Directory

* **What:** Checks for the existence of the `models/` directory and deletes it if present.
* **Why:** To ensure that all models are trained entirely from scratch.
* **Reasoning/Choices:** This safeguard prevents the accidental loading of stale model artifacts or preprocessors from previous interrupted notebook runs, guaranteeing reproducibility.

In [ ]:
MODELS_DIR = ROOT / config["models"]["models_dir"]

# Remove the models directory if it exists to train the models from scratch
if MODELS_DIR.exists():
    shutil.rmtree(MODELS_DIR)

#### 3.2 Training Classification Models

* **What:** Triggers the automated classification training pipeline for all selected algorithms (RF, XGBoost, GBM) and outputs their training durations.
* **Why:** To fit the models specifically optimized for the binary `home_win` target.
* **Reasoning/Choices:** Abstracting the training logic into the `train_all` source module keeps the notebook clean and focuses the narrative directly on evaluation rather than boilerplate `.fit()` code.

In [ ]:
# Log the start of the classification training process
log.info("=" * 50)
log.info("Training — classification (home_win)")
log.info("=" * 50)

# Execute the training pipeline for all models defined in the classification task
# timing_clf returns a DataFrame tracking the duration of each model's training
timing_clf = train_all("classification")

# Display the training duration for each model in a clean, formatted table
display(timing_clf.style.format({"train_time_s": "{:.2f}s"}).hide(axis="index"))

#### 3.3 Training Regression Models

* **What:** Triggers the automated regression training pipeline and outputs training times.
* **Why:** To fit the models specifically optimized for the continuous `point_differential` target.
* **Reasoning/Choices:** Kept consistent with the classification approach to maintain architectural symmetry in the project.

In [ ]:
# Log the start of the regression training process
log.info("=" * 50)
log.info("Training — regression (point_differential)")
log.info("=" * 50)

# Execute the training pipeline for all models defined in the regression task
# timing_reg returns a DataFrame tracking the training duration for each model
timing_reg = train_all("regression")

# Display the formatted training duration table for regression models
display(timing_reg.style.format({"train_time_s": "{:.2f}s"}).hide(axis="index"))

### 4. Classification Evaluation (`home_win`)

* **What:** Evaluates binary home-win predictions using accuracy, F1, ROC-AUC, ROC curves, and confusion matrices.
* **Why:** To measure both threshold-based performance and probability-ranking quality.
* **Reasoning/Choices:** Multiple metrics are needed because sports prediction can value calibrated separation, class balance, and false-positive behavior differently depending on the use case.

#### 4.1 Classification Metrics Table

* **What:** Evaluates classification models on both validation and test sets, generating a comparative table of Accuracy, F1, and AUC scores alongside the calculated delta ($\Delta$).
* **Why:** To quantify the models' predictive power and easily check for overfitting.
* **Reasoning/Choices:** Calculating the delta ($\Delta = test - val$) provides an immediate visual diagnostic cue for performance degradation and generalization capability.

In [ ]:
def eval_clf_split(X_raw, y, task="classification"):
    """Returns a DataFrame with accuracy/F1/AUC metrics for a given split."""
    # Load the fitted preprocessor for the specific task
    prep = joblib.load(MODELS_DIR / f"preprocessor_{task}_v1.joblib")
    # Transform raw input features using the loaded preprocessor
    X = transform(X_raw, prep)
    
    rows = []
    # Iterate through model names to calculate evaluation metrics
    for name in MODEL_NAMES:
        model = joblib.load(MODELS_DIR / f"{name}_{task}_v1.joblib")
        y_pred = model.predict(X)
        y_prob = predict_proba(model, X)
        
        # Append performance metrics for each model
        rows.append({
            "model":    name,
            "accuracy": accuracy_score(y, y_pred),
            "f1":       f1_score(y, y_pred, zero_division=0),
            "auc_roc":  roc_auc_score(y, y_prob),
        })
    return pd.DataFrame(rows).set_index("model")


# Evaluate models on both validation and test sets
val_clf  = eval_clf_split(X_val_clf,  y_val_clf)
test_clf = eval_clf_split(X_test_clf, y_test_clf)

# Combine results into a comparison DataFrame
compare_clf = pd.concat(
    [val_clf.add_suffix("_val"), test_clf.add_suffix("_test")],
    axis=1
)[["accuracy_val", "accuracy_test", "f1_val", "f1_test", "auc_roc_val", "auc_roc_test"]]

# Calculate delta (test - validation) to identify performance drift
compare_clf["Δacc"]     = compare_clf["accuracy_test"] - compare_clf["accuracy_val"]
compare_clf["Δauc_roc"] = compare_clf["auc_roc_test"]  - compare_clf["auc_roc_val"]

# Display the comparison table with highlights
print(f"Baseline (majority class): {BASELINE_ACC:.3f}\n")
display(
    compare_clf.style
    .format("{:.4f}")
    .highlight_max(subset=["accuracy_val", "accuracy_test", "f1_val", "f1_test",
                            "auc_roc_val", "auc_roc_test"], color="#d4edda")
    .highlight_min(subset=["Δacc", "Δauc_roc"], color="#f8d7da")
    .set_caption("Classification — validation vs. test comparison (Δ = test − val)")
)

print("\nInterpretation: Δ << 0 indicates overfitting.")

#### 4.2 ROC Curves

* **What:** Generates and plots ROC (Receiver Operating Characteristic) curves for all classification models evaluated on the test set.
* **Why:** To visualize the trade-off between the True Positive Rate and False Positive Rate across different probability thresholds.
* **Reasoning/Choices:** ROC-AUC is a threshold-invariant metric, making it an excellent tool to evaluate the models' raw probabilistic separation capabilities regardless of the default 0.5 decision boundary.

In [ ]:
# Generate and display the ROC curves for all models evaluated on the test set
# The plot_roc_curves function handles the computation of TPR/FPR and plotting logic
fig = plot_roc_curves("classification")

# Add a title and adjust layout for better visual presentation
fig.suptitle("ROC Curves — Test Set", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

#### 4.3 Confusion Matrices and Diagnostics

* **What:** Displays side-by-side confusion matrices and calculates precision and recall for home-team predictions.
* **Why:** To understand the exact failure modes of each classifier, including whether a model over-predicts home wins.
* **Reasoning/Choices:** Accuracy is useful but incomplete; false positives and false negatives matter in sports analytics because different applications may value precision, recall, or upset detection differently.

In [ ]:
# Load the preprocessor and transform the test set for consistent evaluation across all models
prep = joblib.load(MODELS_DIR / "preprocessor_classification_v1.joblib")
X_test_t = transform(X_test_clf, prep)

# Create a side-by-side subplot figure for comparing confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Iterate through models to generate and plot each confusion matrix
for ax, name, label in zip(axes, MODEL_NAMES, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    y_pred = model.predict(X_test_t)
    cm = confusion_matrix(y_test_clf, y_pred)

    disp = ConfusionMatrixDisplay(cm, display_labels=["Away Win", "Home Win"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(f"{label}\nAccuracy={accuracy_score(y_test_clf, y_pred):.3f}")

# Finalize plot layout
plt.suptitle("Confusion Matrix — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

# Calculate and print specific diagnostic metrics (Precision/Recall) for each model
print("Metrics derived from confusion matrices:\n")
for name, label in zip(MODEL_NAMES, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    y_pred = model.predict(X_test_t)
    tn, fp, fn, tp = confusion_matrix(y_test_clf, y_pred).ravel()
    
    # Precision: How many of the predicted Home Wins were correct?
    # Recall: How many of the actual Home Wins did the model capture?
    precision_home = tp / (tp + fp)
    recall_home = tp / (tp + fn)

    print(f"{label:<18} TN={tn:>4d}  FP={fp:>4d}  FN={fn:>4d}  TP={tp:>4d}  "
          f"Precision home={precision_home:.4f}  Recall home={recall_home:.4f}")

### 5. Regression Evaluation (`point_differential`)

* **What:** Evaluates point-differential predictions using MAE, RMSE, R², scatter plots, residual diagnostics, and derived win labels.
* **Why:** To test whether modeling margin of victory provides richer information than direct binary classification.
* **Reasoning/Choices:** Regression metrics explain spread accuracy, while deriving `home_win` from predicted point differential checks whether that richer target also improves game-outcome prediction.

#### 5.1 Regression Metrics Table

* **What:** Evaluates the regression models, computing Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and $R^2$ against the baseline model.
* **Why:** To quantify how accurately the models predict the actual game spread (margin of victory).
* **Reasoning/Choices:** Utilizing both MAE (robust to outliers) and RMSE (heavily penalizes large errors) provides a balanced view of model accuracy, which is highly relevant given the noisy nature of extreme NBA blowouts.

In [ ]:
def eval_reg_split(X_raw, y, task="regression"):
    """Returns a DataFrame with MAE, RMSE, and R² metrics for a given split."""
    # Load the fitted preprocessor for the regression task
    prep = joblib.load(MODELS_DIR / f"preprocessor_{task}_v1.joblib")
    # Transform raw input features
    X = transform(X_raw, prep)
    
    rows = []
    # Calculate performance metrics for each model
    for name in MODEL_NAMES:
        model = joblib.load(MODELS_DIR / f"{name}_{task}_v1.joblib")
        y_pred = model.predict(X)
        rmse = float(np.sqrt(mean_squared_error(y, y_pred)))
        rows.append({
            "model": name,
            "mae":   mean_absolute_error(y, y_pred),
            "rmse":  rmse,
            "r2":    r2_score(y, y_pred),
        })
    return pd.DataFrame(rows).set_index("model")


# Perform evaluations for ML models
val_reg = eval_reg_split(X_val_reg, y_val_reg)
test_reg = eval_reg_split(X_test_reg, y_test_reg)

# Build baseline metrics directly using the global arrays computed in Section 2
baseline_reg = pd.DataFrame(
    {
        "mae_val": [mean_absolute_error(y_val_reg, y_pred_val_base)],
        "mae_test": [mean_absolute_error(y_test_reg, y_pred_test_base)],
        "rmse_val": [float(np.sqrt(mean_squared_error(y_val_reg, y_pred_val_base)))],
        "rmse_test": [float(np.sqrt(mean_squared_error(y_test_reg, y_pred_test_base)))],
        "r2_val": [r2_score(y_val_reg, y_pred_val_base)],
        "r2_test": [r2_score(y_test_reg, y_pred_test_base)],
    },
    index=["baseline_mean_train"],
)

# Consolidate results for comparison
compare_reg = pd.concat(
    [val_reg.add_suffix("_val"), test_reg.add_suffix("_test")],
    axis=1
)[["mae_val", "mae_test", "rmse_val", "rmse_test", "r2_val", "r2_test"]]

compare_reg = pd.concat([compare_reg, baseline_reg])

# Calculate performance deltas (test - validation)
compare_reg["Δmae"] = compare_reg["mae_test"]  - compare_reg["mae_val"]
compare_reg["Δr2"]  = compare_reg["r2_test"]   - compare_reg["r2_val"]

# Display the comparison table
display(
    compare_reg.style
    .format("{:.4f}")
    .highlight_min(subset=["mae_val", "mae_test", "rmse_val", "rmse_test"], color="#d4edda")
    .highlight_max(subset=["r2_val",  "r2_test"],  color="#d4edda")
    .highlight_max(subset=["Δmae"],  color="#f8d7da")
    .set_caption("Regression — validation vs. test comparison with baseline (Δ = test − val)")
)

#### 5.2 Scatter Plots: Predicted vs Actual

* **What:** Plots predicted point differentials against actual point differentials with a perfect-prediction reference line.
* **Why:** To visually inspect correlation, spread, and regression range compression.
* **Reasoning/Choices:** Low-opacity points reveal density while preserving outliers, which is important because NBA blowouts can expose whether models under-predict extreme margins.

In [ ]:
# Load the regression preprocessor and transform the test set for consistent evaluation
prep_reg = joblib.load(MODELS_DIR / "preprocessor_regression_v1.joblib")
X_test_reg_t = transform(X_test_reg, prep_reg)

# Create a subplot figure for side-by-side comparison of regression performance
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

# Iterate through models to visualize predicted versus actual values
for ax, name, color, label in zip(axes, MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    y_pred = model.predict(X_test_reg_t)
    r2 = r2_score(y_test_reg, y_pred)
    mae = mean_absolute_error(y_test_reg, y_pred)

    # Plot actual vs predicted values with transparency for density
    ax.scatter(y_test_reg, y_pred, alpha=0.25, s=8, color=color, rasterized=True)
    
    # Draw a diagonal reference line representing perfect predictions
    lim = max(abs(y_test_reg.min()), abs(y_test_reg.max())) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1, label="Perfect")
    
    # Configure axes limits and labels for readability
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("Actual point_differential")
    ax.set_ylabel("Predicted" if ax is axes[0] else "")
    ax.set_title(f"{label}\nR²={r2:.3f}  MAE={mae:.2f}")
    ax.set_aspect("equal")

# Set the overall figure title and adjust layout
fig.suptitle("Scatter Predicted vs Actual — Test Set (Regression)", fontsize=13)
plt.tight_layout()
plt.show()

#### 5.3 Residual Distributions

* **What:** Generates residual histograms and Q-Q plots for each regression model.
* **Why:** To diagnose bias, heavy tails, and systematic spread errors after point-differential prediction.
* **Reasoning/Choices:** Residuals centered near zero indicate limited average bias, while skewness or fat tails reveal game contexts where the model consistently misses margin size.

In [ ]:
# Create a 2x3 grid for residual analysis (histograms and Q-Q plots)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for col, name, color, label in zip(range(3), MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    # Load the trained model and calculate residuals on the test set
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    y_pred = model.predict(X_test_reg_t)
    residuals = y_pred - y_test_reg.values

    # Top row: Histogram of residuals vs. theoretical normal distribution
    ax_hist = axes[0][col]
    ax_hist.hist(residuals, bins=60, density=True, color=color, alpha=0.7, edgecolor="none")
    mu, sigma = residuals.mean(), residuals.std()
    x_range = np.linspace(residuals.min(), residuals.max(), 300)
    ax_hist.plot(x_range, stats.norm.pdf(x_range, mu, sigma), "k-", lw=1.5, label=f"N({mu:.1f},{sigma:.1f})")
    ax_hist.axvline(0, color="red", lw=1.2, linestyle="--")
    ax_hist.set_title(f"{label}\nresidual — mean={mu:+.2f}  std={sigma:.2f}")
    ax_hist.set_xlabel("Residual (pred − actual)")
    ax_hist.legend(fontsize=8)

    # Bottom row: Q-Q plot to assess normality of residuals
    ax_qq = axes[1][col]
    (osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist="norm")
    ax_qq.scatter(osm, osr, alpha=0.3, s=6, color=color, rasterized=True)
    ax_qq.plot(osm, slope * np.array(osm) + intercept, "k-", lw=1.5)
    ax_qq.set_title(f"Q-Q  (R={r:.3f})")
    ax_qq.set_xlabel("Theoretical Quantiles")
    ax_qq.set_ylabel("Sample Quantiles")

# Set figure title and finalize layout
fig.suptitle("Residual Distribution — Test Set (Regression)", fontsize=13)
plt.tight_layout()
plt.show()

#### 5.4 Derived Classification vs Direct Classification

* **What:** Converts predicted point differential into a binary home-win prediction and compares it with direct classifiers.
* **Why:** To test whether spread modeling can also solve the win/loss task effectively.
* **Reasoning/Choices:** Direct classifiers often perform better for pure outcome prediction, while regression models provide richer margin information for interpreting game strength.

In [ ]:
# Compare the accuracy of deriving a "win" from regression predictions versus direct classification

print("Derived accuracy (reg pred > 0 ↦ home_win=1)  vs.  direct classification\n")
print(f"{'Model':<22} {'Derived Acc':>14} {'Direct Clf Acc':>16}")
print("-" * 56)

# Load the preprocessor for the classification task
prep_clf = joblib.load(MODELS_DIR / "preprocessor_classification_v1.joblib")
X_test_clf_t = transform(X_test_clf, prep_clf)

for name, label in zip(MODEL_NAMES, MODEL_LABELS):
    # Derived from Regression: Predict win if point differential > 0
    model_reg   = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    y_pred_reg  = model_reg.predict(X_test_reg_t)
    y_win_from_reg = (y_pred_reg > 0).astype(int)
    acc_derived = accuracy_score(y_test_clf, y_win_from_reg)

    # Direct Classification: Predict win using the classification model
    model_clf   = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    y_pred_clf  = model_clf.predict(X_test_clf_t)
    acc_direct  = accuracy_score(y_test_clf, y_pred_clf)

    # Calculate difference between approaches
    delta = acc_direct - acc_derived
    print(f"{label:<22}  {acc_derived:>12.4f}  {acc_direct:>14.4f}   Δ={delta:+.4f}")

# Compare against the baseline
print(f"{'Baseline (majority)':<22}  {BASELINE_ACC:>12.3f}")
print("\n→ If Δ > 0, direct classification is preferable to derived regression.")

### 6. Feature Importance

* **What:** Compares the most influential predictors across classification and regression models.
* **Why:** To interpret which engineered features drive decisions and whether importance patterns are consistent across algorithms.
* **Reasoning/Choices:** Tree-based importance is model-specific, so cross-model agreement is treated as stronger evidence than a single algorithm's ranking.

#### 6.1 Classification - Feature Importance

* **What:** Extracts and visualizes the Top-20 most important features for each classification model.
* **Why:** To identify which specific variables drive the models' decision-making processes.
* **Reasoning/Choices:** Utilizing side-by-side horizontal bar charts allows for rapid visual comparison of feature dominance across different tree-based algorithmic architectures.

In [ ]:
# Create a figure for side-by-side comparison of Top-20 feature importance
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

importance_dfs = {}

# Iterate through each classification model to extract and plot feature importance
for ax, name, color, label in zip(axes, MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    prep  = joblib.load(MODELS_DIR / "preprocessor_classification_v1.joblib")
    feat_names = prep.feature_cols

    # Extract feature importance values and identify the top 20
    importances = model.feature_importances_
    top20_idx   = np.argsort(importances)[::-1][:20]
    top20_names = [feat_names[i] for i in top20_idx]
    top20_vals  = importances[top20_idx]

    # Store full importance mapping for further analysis
    importance_dfs[name] = pd.Series(importances, index=feat_names, name=name).sort_values(ascending=False)

    # Horizontal bar plot for the top 20 features
    ax.barh(range(20), top20_vals[::-1], color=color, alpha=0.85)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top20_names[::-1], fontsize=8)
    ax.set_xlabel("Importance")
    ax.set_title(label, fontsize=11)
    # Align visually to show highest importance closest to the right
    ax.invert_xaxis()

fig.suptitle("Top-20 Feature Importance — Classification", fontsize=13)
plt.tight_layout()
plt.show()

#### 6.2 Classification Global Feature Ranking

* **What:** Aggregates, averages, and ranks feature importance scores across all three classification models, outputting a consensus heatmap.
* **Why:** To establish a globally agreed-upon consensus of the most predictive features in the dataset.
* **Reasoning/Choices:** Utilizing a rank-based system normalizes the fundamentally different importance calculation scales (e.g., Gini Impurity vs. Information Gain) native to Random Forest and XGBoost.

In [ ]:
# Create a DataFrame combining all model importances
all_imp = pd.DataFrame(importance_dfs)
all_imp.columns = MODEL_LABELS

# Calculate the rank for each feature per model (1 = most important)
# A lower rank indicates higher relative importance
rank_df = all_imp.rank(ascending=False, method="min").astype(int)
rank_df["mean_rank"] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values("mean_rank")

print("Top-15 features by average rank across the three classification models\n")
# Display the ranked features with a color gradient highlighting top performers
display(
    rank_df.head(15).style
    .background_gradient(cmap="YlOrRd_r", subset=MODEL_LABELS, vmin=1, vmax=30)
    .format("{:.1f}", subset=["mean_rank"])
    .set_caption("Rank 1 = most important feature in the model")
)

# Check the importance rank of 'is_covid_bubble' specifically
if "is_covid_bubble" in rank_df.index:
    covid_row = rank_df.loc[["is_covid_bubble"]]
    print(f"\nis_covid_bubble — rank across the three models:")
    display(covid_row.style.format("{:.0f}", subset=MODEL_LABELS).format("{:.1f}", subset=["mean_rank"]))
else:
    print("\nis_covid_bubble is not present in the transformed feature set.")

#### 6.3 Regression - Feature Importance

* **What:** Extracts and visualizes the Top-20 most important features for each regression model.
* **Why:** To identify which variables specifically drive the prediction of the margin of victory.
* **Reasoning/Choices:** Maintained for comparative symmetry with the classification task analysis.

In [ ]:
# Create a figure for side-by-side comparison of Top-20 feature importance for Regression
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

importance_reg_dfs = {}
prep_reg_imp = joblib.load(MODELS_DIR / "preprocessor_regression_v1.joblib")
feat_names_reg = prep_reg_imp.feature_cols

# Iterate through regression models to visualize Top-20 feature importance
for ax, name, color, label in zip(axes, MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")

    importances = model.feature_importances_
    top20_idx   = np.argsort(importances)[::-1][:20]
    top20_names = [feat_names_reg[i] for i in top20_idx]
    top20_vals  = importances[top20_idx]

    importance_reg_dfs[name] = pd.Series(importances, index=feat_names_reg, name=name).sort_values(ascending=False)

    ax.barh(range(20), top20_vals[::-1], color=color, alpha=0.85)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top20_names[::-1], fontsize=8)
    ax.set_xlabel("Importance")
    ax.set_title(label, fontsize=11)
    ax.invert_xaxis()

fig.suptitle("Top-20 Feature Importance — Regression", fontsize=13)
plt.tight_layout()
plt.show()

#### 6.4 Regression Global Feature Ranking

* **What:** Aggregates and ranks feature importance globally across all three regression models.
* **Why:** To find consensus features explicitly tailored to forecasting point spreads.
* **Reasoning/Choices:** Provides a normalized foundation to compare against the classification task.

In [ ]:
# Calculate and rank feature importance for Regression
all_imp_reg = pd.DataFrame(importance_reg_dfs)
all_imp_reg.columns = MODEL_LABELS

rank_reg_df = all_imp_reg.rank(ascending=False, method="min").astype(int)
rank_reg_df["mean_rank"] = rank_reg_df.mean(axis=1)
rank_reg_df = rank_reg_df.sort_values("mean_rank")

print("Top-15 features by average rank across the three regression models\n")
display(
    rank_reg_df.head(15).style
    .background_gradient(cmap="YlOrRd_r", subset=MODEL_LABELS, vmin=1, vmax=30)
    .format("{:.1f}", subset=["mean_rank"])
    .set_caption("Rank 1 = most important feature in the model")
)

#### 6.5 Classification vs Regression

* **What:** Merges feature rankings from both tasks and computes the absolute difference ($\Delta$) to highlight features that shift in importance.
* **Why:** To determine if predicting the margin of victory inherently relies on different statistical factors than simply predicting a binary win.
* **Reasoning/Choices:** Highlighting the absolute rank delta reveals critical domain insights (e.g., identifying if team pace matters significantly more for predicting the spread than the raw game outcome).

In [ ]:
# Compare feature rankings between Classification and Regression tasks
rank_task_compare = pd.DataFrame(
    {
        "rank_classification": rank_df["mean_rank"],
        "rank_regression": rank_reg_df["mean_rank"],
    }
).dropna()

# Calculate the delta between task ranks to see which features shift importance
rank_task_compare["Δrank_reg_minus_clf"] = (
    rank_task_compare["rank_regression"] - rank_task_compare["rank_classification"]
)
rank_task_compare["abs_Δrank"] = rank_task_compare["Δrank_reg_minus_clf"].abs()
rank_task_compare = rank_task_compare.sort_values("abs_Δrank", ascending=False)

print("Top-20 features with largest average rank difference between classification and regression\n")
display(
    rank_task_compare.head(20).style
    .format("{:.1f}")
    .background_gradient(cmap="RdYlGn_r", subset=["rank_classification", "rank_regression"], vmin=1, vmax=len(FEATURE_COLS))
    .background_gradient(cmap="Blues", subset=["abs_Δrank"])
    .set_caption("Δ > 0 = more important in classification; Δ < 0 = more important in regression")
)

### 7. Error Analysis

* **What:** Investigates misclassified games, large regression errors, and error rates across schedule or season contexts.
* **Why:** To identify where models fail and whether those failures reflect random close games or systematic blind spots.
* **Reasoning/Choices:** Error segmentation is necessary because aggregate metrics can hide weaknesses around back-to-backs, anomalous seasons, or upset-heavy matchup contexts.

#### 7.1 Error Analysis Data Setup

* **What:** Merges model predictions with raw game metadata (season, dates, team abbreviations, COVID status) to build a unified diagnostic DataFrame.
* **Why:** To prepare the dataset for granular, contextual inspection of failure points.
* **Reasoning/Choices:** Generating an `all_wrong` flag tracks games missed by *all three* models simultaneously, effectively filtering out model-specific variance to isolate games that were fundamentally unpredictable based on the provided features.

In [ ]:
# Reload the full CSV to include metadata (season, date, B2B, COVID, etc.) for result analysis
full_df   = pd.read_csv(FEATURES_PATH)
test_meta = full_df[full_df["split"] == "test"].copy().reset_index(drop=True)

# Generate predictions for all three classifiers on the test set
pred_clf = {}
prob_clf = {}
for name in MODEL_NAMES:
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    pred_clf[name] = model.predict(X_test_clf_t)
    prob_clf[name] = predict_proba(model, X_test_clf_t)

# Generate predictions for all three regressors on the test set
pred_reg = {}
for name in MODEL_NAMES:
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    pred_reg[name] = model.predict(X_test_reg_t)

# Consolidate results into a unified DataFrame for inspection
test_results = test_meta[[
    "game_date", "season",
    "home_team_abbreviation", "away_team_abbreviation",
    "home_wl", "away_wl", "home_pts", "away_pts",
    "is_covid_bubble",
    "home_is_back_to_back", "away_is_back_to_back",
]].copy()

# Add ground truth and actual point spread
test_results["y_true_clf"]   = y_test_clf.values
test_results["y_true_reg"]   = y_test_reg.values
test_results["actual_spread"] = test_results["home_pts"] - test_results["away_pts"]

# Map predictions to the results DataFrame using simplified model keys
for name in MODEL_NAMES:
    short = name.replace("random_forest", "rf").replace("gradient_boosting", "gbm").replace("xgboost", "xgb")
    test_results[f"pred_clf_{short}"] = pred_clf[name]
    test_results[f"pred_reg_{short}"] = pred_reg[name]

# Identify erroneous predictions for each classifier
wrong_rf  = test_results["pred_clf_rf"]  != test_results["y_true_clf"]
wrong_xgb = test_results["pred_clf_xgb"] != test_results["y_true_clf"]
wrong_gbm = test_results["pred_clf_gbm"] != test_results["y_true_clf"]

# Aggregate error counts
test_results["all_wrong"] = wrong_rf & wrong_xgb & wrong_gbm
test_results["n_wrong"]   = wrong_rf.astype(int) + wrong_xgb.astype(int) + wrong_gbm.astype(int)

# Display error summary
n_all_wrong = test_results["all_wrong"].sum()
print(f"Total games in test set     : {len(test_results):>6}")
print(f"Missed by all three models  : {n_all_wrong:>6}  ({n_all_wrong/len(test_results)*100:.1f}%)")
print(f"Missed by at least one model: {(test_results['n_wrong']>0).sum():>6}")

#### 7.2 Analyzing Universal Upsets

* **What:** Analyzes the statistical distribution and specific instances of games misclassified by the consensus of all three models.
* **Why:** To identify qualitative patterns in "upsets" or fundamentally unpredictable games.
* **Reasoning/Choices:** Comparing the actual absolute margins of misclassified games against correctly classified games helps prove whether these failures were tight "coin-flip" games or total structural blind spots.

In [ ]:
# Analyze characteristics of games misclassified by all three models
misclassified = test_results[test_results["all_wrong"]].copy()
correct = test_results[~test_results["all_wrong"]].copy()

print("=" * 60)
print("Games misclassified by RF + XGBoost + GBM")
print("=" * 60)
print(f"Distribution of y_true (home_win):")
print(misclassified["y_true_clf"].value_counts().rename({0: "Away Win", 1: "Home Win"}).to_string())

print(f"\nActual absolute spread (margin of victory):")
print(f"  Misclassified: mean={misclassified['actual_spread'].abs().mean():.2f}  "
      f"median={misclassified['actual_spread'].abs().median():.2f}")
print(f"  Correct      : mean={correct['actual_spread'].abs().mean():.2f}  "
      f"median={correct['actual_spread'].abs().median():.2f}")

print("\nNote: Misclassified games tend to have tighter margins, indicating genuine upsets.")

# Top 10 most "surprising" games (smallest spread among misclassified)
print("\nTop 10 upsets (smallest absolute spread) among games misclassified by all models:")
cols_show = ["game_date", "season", "home_team_abbreviation", "away_team_abbreviation",
             "home_pts", "away_pts", "actual_spread", "y_true_clf"]
display(
    misclassified.sort_values("actual_spread", key=abs).head(10)[cols_show]
    .rename(columns={"y_true_clf": "home_win", "actual_spread": "spread"})
    .style.hide(axis="index")
)

# Histogram comparison of spreads
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(correct["actual_spread"].abs(),       bins=40, alpha=0.6, label="Correct",  color="steelblue", density=True)
ax.hist(misclassified["actual_spread"].abs(), bins=40, alpha=0.6, label="Wrong (all 3)", color="tomato",     density=True)
ax.set_xlabel("|Spread| actual (points)")
ax.set_ylabel("Density")
ax.set_title("Spread distribution — correct vs. misclassified games")
ax.legend()
plt.tight_layout()
plt.show()

#### 7.3 Analyzing Massive Regression Errors

* **What:** Computes the Mean Absolute Error across regression models and filters for games exceeding a predefined blowout threshold (20+ points).
* **Why:** To flag and inspect anomalous blowouts or massive upsets where the model failed spectacularly.
* **Reasoning/Choices:** Setting a hard, domain-specific blowout threshold helps isolate historically anomalous events heavily influenced by unpredictable real-world factors (e.g., sudden mid-game injuries or drastic fourth-quarter benching).

In [ ]:
# Calculate the absolute error (AE) for each regression model
for name in MODEL_NAMES:
    short = name.replace("random_forest", "rf").replace("gradient_boosting", "gbm").replace("xgboost", "xgb")
    test_results[f"ae_{short}"] = (test_results[f"pred_reg_{short}"] - test_results["y_true_reg"]).abs()

# Compute the mean absolute error across the three models
test_results["mean_ae"] = test_results[["ae_rf", "ae_xgb", "ae_gbm"]].mean(axis=1)

# Define and identify games where models struggled significantly (Blowout Threshold)
BLOWOUT_THRESHOLD = 20  
large_errors = test_results[test_results["mean_ae"] >= BLOWOUT_THRESHOLD].copy()
print(f"NBA blowout threshold for mean MAE: {BLOWOUT_THRESHOLD:.0f} points")
print(f"Games with high error (>= {BLOWOUT_THRESHOLD:.0f} points): {len(large_errors)} ({len(large_errors)/len(test_results)*100:.1f}%)\n")

# Display the top 10 games with the largest mean regression error
cols_reg = ["game_date", "season", "home_team_abbreviation", "away_team_abbreviation",
            "actual_spread", "pred_reg_rf", "pred_reg_xgb", "pred_reg_gbm", "mean_ae"]
print("Top 10 games with the highest mean regression error:")
display(
    large_errors.nlargest(10, "mean_ae")[cols_reg]
    .round(1).style.hide(axis="index")
    .set_caption("Actual spread vs. model predictions")
)

print("\nNote: Historically anomalous margins (blowouts or massive upsets) contribute to")
print("      the tail of the error distribution. Check for potential injuries or schedule-related rest.")

#### 7.4 Segmenting Errors by Context

* **What:** Segments and visualizes model error rates across historical NBA seasons and by Back-to-Back (B2B) schedule constraints.
* **Why:** To uncover systemic blind spots and ensure the model does not unfairly penalize specific game contexts.
* **Reasoning/Choices:** Schedule fatigue (B2B) and evolving seasonal meta-games are classic confounders in NBA analytics. Verifying error rates across these dimensions guarantees the robustness and contextual fairness of the model.

In [ ]:
# Create a figure for segmented error analysis: Seasonality and Back-to-Back schedules
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# ── Seasonal Error Analysis ──────────────────────────────────────────────
ax = axes[0]
err_by_season = (
    test_results.groupby("season")["all_wrong"]
    .mean()
    .reset_index()
    .rename(columns={"all_wrong": "err_rate_all3"})
)
ax.bar(range(len(err_by_season)), err_by_season["err_rate_all3"], color="steelblue", alpha=0.8)
ax.set_xticks(range(len(err_by_season)))
ax.set_xticklabels(err_by_season["season"].str[-5:], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Error Rate (all three models)")
ax.set_title("Errors by Season")

# ── Back-to-Back (B2B) Schedule Impact ───────────────────────────────────
ax = axes[1]
test_results["b2b_flag"] = (
    test_results["home_is_back_to_back"].astype(bool) |
    test_results["away_is_back_to_back"].astype(bool)
)
b2b_err = test_results.groupby("b2b_flag")["all_wrong"].mean().reset_index()
b2b_err["label"] = b2b_err["b2b_flag"].map({False: "No B2B", True: "At least 1 B2B"})
ax.bar(b2b_err["label"], b2b_err["all_wrong"], color=["steelblue", "darkorange"], alpha=0.85)
ax.set_ylabel("Error Rate (all three models)")
ax.set_title("Errors: Back-to-Back Impact")

# Add text labels on the bars for clarity
for i, (_, row) in enumerate(b2b_err.iterrows()):
    ax.text(i, row["all_wrong"] + 0.002, f"{row['all_wrong']:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

# Print summary table
print("Average error rate per segment:\n")
print(f"{'Segment':<30} {'Error Rate (all 3)':>22}")
print("-" * 56)
for _, row in b2b_err.iterrows():
    print(f"{row['label']:<30}  {row['all_wrong']:>20.3f}")

### 8. Conclusion

* **What:** Dynamically summarizes the best cross-validation model, final test-set performance, leading features, and major error patterns.
* **Why:** To make the notebook conclusion reflect the actual metrics generated during the current run.
* **Reasoning/Choices:** A dynamic conclusion is appropriate here because model rankings, feature importances, and upset rates can change when the dataset, season range, or training pipeline changes.

In [ ]:
# Extracting dynamic metrics for the report
best_cv_model = summary_df["accuracy_mean"].idxmax()
best_clf_model = compare_clf["accuracy_test"].idxmax()
best_clf_acc = compare_clf.loc[best_clf_model, "accuracy_test"]
best_reg_model = compare_reg.loc[["random_forest", "xgboost", "gradient_boosting"], "mae_test"].idxmin()
top_clf_feature = rank_df.index[0]
top_reg_feature = rank_reg_df.index[0]
upset_percentage = n_all_wrong / len(test_results) * 100


In [ ]:

dynamic_markdown = f"""

This section dynamically summarizes the results based on the current dataset execution.

### 8.1 Dataset & Baselines
The classification baseline (predicting the majority class) is **{BASELINE_ACC:.1%}**. The best performing classification model on the test set is **{best_clf_model.replace('_', ' ').title()}**, achieving an accuracy of **{best_clf_acc:.1%}**. This confirms that the engineered features provide significant predictive signal beyond standard home-court advantage.

### 8.2 Walk-Forward Validation
Time-series aware cross-validation indicates that **{best_cv_model.replace('_', ' ').title()}** provides the most stable performance across historical seasons. Monitoring the standard deviation (σ) across these folds ensures the model is learning generalized basketball patterns rather than overfitting to specific eras or schedule anomalies.

### 8.3 & 8.4 Classification Performance
Based on the Test-Validation Delta (Δ), the models show varying degrees of generalization. Currently, **{best_clf_model.replace('_', ' ').title()}** is recommended for the binary `home_win` prediction task, balancing raw accuracy, F1 score, and ROC-AUC stability. 

### 8.5 Regression Performance
For predicting the point differential (spread), **{best_reg_model.replace('_', ' ').title()}** achieved the lowest Mean Absolute Error (MAE) on the test set. While regression is inherently noisier due to late-game variance (garbage time, intentional fouling), it provides a valuable secondary confidence metric. However, if the derived win/loss accuracy from regression underperforms direct classification, the direct classifier should remain the primary decision engine.

### 8.6 Feature Importance Consensus
The feature ranking across models reveals that the most critical predictor for game outcomes is currently **`{top_clf_feature}`**, while spread prediction relies heavily on **`{top_reg_feature}`**. The dominance of rolling and differential stats over static contextual flags validates the assumption that recent momentum and relative team strength dictate NBA outcomes more than schedule structure alone.

### 8.7 & 8.8 Error & Context Analysis
Approximately **{upset_percentage:.1f}%** of test games were misclassified by *all three* models. These "universal misses" generally exhibit tighter actual point spreads, representing genuine NBA upsets or coin-flip games. Furthermore, the blowout threshold analysis isolates games where regression models failed spectacularly, highlighting the limitations of statistical models against unpredictable real-world events (e.g., in-game injuries).

### 8.9 Final Model Selection
**Automated Recommendation:** For production deployment based on this specific dataset run:
* **Primary Classifier:** Use `{best_clf_model}` for official win/loss predictions.
* **Complementary Regressor:** Use `{best_reg_model}` for spread estimation and confidence scaling.
"""

display(Markdown(dynamic_markdown))